In [0]:
from pyspark.sql.functions import (
    col, year, month, dayofmonth, dayofweek, quarter,
    date_format, monotonically_increasing_id, current_timestamp,
    sum as _sum, count as _count, avg as _avg, max as _max,
    round as _round, lit, when, concat_ws
)
from pyspark.sql.types import IntegerType, DateType

In [0]:
# ===========================================================================
# CÉLULA 1 - Configurações
# ===========================================================================
CATALOG       = "workspace"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD   = "gold"

In [0]:
# ===========================================================================
# CÉLULA 2 - Criar schema GOLD
# ===========================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")
print(f"Schema '{SCHEMA_GOLD}' pronto.")


In [0]:
# ===========================================================================
# CÉLULA 3 - DIM_CLIENTE
# ===========================================================================
print("\n>>> Criando dim_cliente")

df_dim_cliente = spark.sql(f"""
    SELECT
        id_cliente                          AS sk_cliente,
        id_cliente                          AS nk_cliente,
        nome                                AS nome_cliente,
        email,
        COALESCE(telefone, 'Não informado') AS telefone,
        COALESCE(cidade,   'Não informado') AS cidade,
        COALESCE(estado,   'XX')            AS estado,
        CASE estado
            WHEN 'SP' THEN 'Sudeste'
            WHEN 'RJ' THEN 'Sudeste'
            WHEN 'MG' THEN 'Sudeste'
            WHEN 'ES' THEN 'Sudeste'
            WHEN 'RS' THEN 'Sul'
            WHEN 'SC' THEN 'Sul'
            WHEN 'PR' THEN 'Sul'
            WHEN 'GO' THEN 'Centro-Oeste'
            WHEN 'MT' THEN 'Centro-Oeste'
            WHEN 'MS' THEN 'Centro-Oeste'
            WHEN 'DF' THEN 'Centro-Oeste'
            WHEN 'BA' THEN 'Nordeste'
            WHEN 'CE' THEN 'Nordeste'
            WHEN 'PE' THEN 'Nordeste'
            WHEN 'AM' THEN 'Norte'
            WHEN 'PA' THEN 'Norte'
            ELSE 'Outras'
        END                                 AS regiao,
        data_cadastro,
        ativo,
        current_timestamp()                 AS _gold_ts
    FROM {CATALOG}.{SCHEMA_SILVER}.clientes
""")

df_dim_cliente.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.dim_cliente")

print(f"    dim_cliente: {df_dim_cliente.count()} registros")

In [0]:
# ===========================================================================
# CÉLULA 4 - DIM_PRODUTO
# ===========================================================================
print("\n>>> Criando dim_produto")

df_dim_produto = spark.sql(f"""
    SELECT
        p.id_produto                        AS sk_produto,
        p.id_produto                        AS nk_produto,
        p.nome                              AS nome_produto,
        c.id_categoria                      AS sk_categoria,
        c.nome                              AS categoria,
        p.preco                             AS preco_atual,
        CASE
            WHEN p.preco <  100  THEN 'Econômico'
            WHEN p.preco <  500  THEN 'Intermediário'
            WHEN p.preco < 2000  THEN 'Premium'
            ELSE 'Luxo'
        END                                 AS faixa_preco,
        p.estoque,
        p.ativo,
        current_timestamp()                 AS _gold_ts
    FROM {CATALOG}.{SCHEMA_SILVER}.produtos   p
    JOIN {CATALOG}.{SCHEMA_SILVER}.categorias c
      ON p.id_categoria = c.id_categoria
""")

df_dim_produto.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.dim_produto")

print(f"    dim_produto: {df_dim_produto.count()} registros")

In [0]:
# ===========================================================================
# CÉLULA 5 - DIM_DATA (gerada a partir dos pedidos existentes)
# ===========================================================================
print("\n>>> Criando dim_data")

df_dim_data = spark.sql(f"""
    SELECT DISTINCT
        CAST(DATE_FORMAT(data_pedido, 'yyyyMMdd') AS INT)  AS sk_data,
        CAST(data_pedido AS DATE)                          AS data_completa,
        YEAR(data_pedido)                                  AS ano,
        QUARTER(data_pedido)                               AS trimestre,
        MONTH(data_pedido)                                 AS mes,
        DAY(data_pedido)                                   AS dia,
        DAYOFWEEK(data_pedido)                             AS dia_semana_num,
        DATE_FORMAT(data_pedido, 'EEEE')                   AS dia_semana_nome,
        DATE_FORMAT(data_pedido, 'MMMM')                   AS mes_nome,
        CASE WHEN DAYOFWEEK(data_pedido) IN (1,7)
             THEN 'Fim de semana' ELSE 'Dia útil'
        END                                                AS tipo_dia,
        CONCAT('T', QUARTER(data_pedido))                  AS trimestre_label,
        current_timestamp()                                AS _gold_ts
    FROM {CATALOG}.{SCHEMA_SILVER}.pedidos
    WHERE data_pedido IS NOT NULL
""")

df_dim_data.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.dim_data")

print(f"    dim_data: {df_dim_data.count()} registros")

In [0]:
# ===========================================================================
# CÉLULA 6 - DIM_PAGAMENTO
# ===========================================================================
print("\n>>> Criando dim_pagamento")

df_dim_pagamento = spark.sql(f"""
    SELECT
        ROW_NUMBER() OVER (ORDER BY forma_pagto)           AS sk_pagamento,
        forma_pagto                                        AS codigo_pagamento,
        CASE forma_pagto
            WHEN 'pix'            THEN 'PIX'
            WHEN 'cartao_credito' THEN 'Cartão de Crédito'
            WHEN 'cartao_debito'  THEN 'Cartão de Débito'
            WHEN 'boleto'         THEN 'Boleto Bancário'
            ELSE 'Outros'
        END                                                AS descricao_pagamento,
        CASE forma_pagto
            WHEN 'pix'            THEN 'Instantâneo'
            WHEN 'cartao_credito' THEN 'Parcelável'
            WHEN 'cartao_debito'  THEN 'À vista'
            WHEN 'boleto'         THEN 'À vista'
            ELSE 'Outros'
        END                                                AS tipo_pagamento,
        current_timestamp()                                AS _gold_ts
    FROM (
        SELECT DISTINCT forma_pagto
        FROM {CATALOG}.{SCHEMA_SILVER}.pedidos
        WHERE forma_pagto IS NOT NULL
    )
""")

df_dim_pagamento.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.dim_pagamento")

print(f"    dim_pagamento: {df_dim_pagamento.count()} registros")

In [0]:
# ===========================================================================
# CÉLULA 7 - FATO_VENDAS (tabela fato principal)
# ===========================================================================
print("\n>>> Criando fato_vendas")

df_fato = spark.sql(f"""
    SELECT
        -- Surrogate Key da fato
        i.id_item                                          AS sk_venda,

        -- Foreign Keys para as dimensões
        p.id_pedido                                        AS nk_pedido,
        CAST(DATE_FORMAT(p.data_pedido,'yyyyMMdd') AS INT) AS sk_data,
        c.id_cliente                                       AS sk_cliente,
        i.id_produto                                       AS sk_produto,
        pg.sk_pagamento,

        -- Medidas / Métricas
        i.quantidade,
        i.preco_unit                                       AS preco_unitario,
        i.valor_item                                       AS valor_bruto,
        ROUND(i.valor_item * 0.1, 2)                       AS valor_desconto,
        ROUND(i.valor_item * 0.9, 2)                       AS valor_liquido,
        ROUND(i.valor_item * 0.15, 2)                      AS valor_margem,

        -- Atributos degenerados da fato
        p.status                                           AS status_pedido,
        YEAR(p.data_pedido)                                AS ano_pedido,
        MONTH(p.data_pedido)                               AS mes_pedido,

        current_timestamp()                                AS _gold_ts

    FROM {CATALOG}.{SCHEMA_SILVER}.itens_pedido   i
    JOIN {CATALOG}.{SCHEMA_SILVER}.pedidos         p  ON i.id_pedido = p.id_pedido
    JOIN {CATALOG}.{SCHEMA_SILVER}.clientes        c  ON p.id_cliente = c.id_cliente
    JOIN {CATALOG}.{SCHEMA_GOLD}.dim_pagamento     pg ON p.forma_pagto = pg.codigo_pagamento
    -- Só inclui pedidos não cancelados
    WHERE p.status <> 'cancelado'
""")

df_fato.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.fato_vendas")

print(f"    fato_vendas: {df_fato.count()} registros")


In [0]:
# ===========================================================================
# CÉLULA 8 - ANÁLISE: Top produtos por receita (validação)
# ===========================================================================
print("\n>>> Análise: Top produtos por receita")
display(spark.sql(f"""
    SELECT
        dp.nome_produto,
        dp.categoria,
        SUM(fv.quantidade)   AS total_quantidade,
        SUM(fv.valor_bruto)  AS receita_bruta,
        SUM(fv.valor_liquido)AS receita_liquida
    FROM {CATALOG}.{SCHEMA_GOLD}.fato_vendas    fv
    JOIN {CATALOG}.{SCHEMA_GOLD}.dim_produto    dp ON fv.sk_produto  = dp.sk_produto
    GROUP BY dp.nome_produto, dp.categoria
    ORDER BY receita_bruta DESC
    LIMIT 10
"""))

In [0]:
# ===========================================================================
# CÉLULA 9 - ANÁLISE: Receita por mês e forma de pagamento
# ===========================================================================
print("\n>>> Análise: Receita por mês × forma de pagamento")
display(spark.sql(f"""
    SELECT
        dd.ano,
        dd.mes_nome,
        dpg.descricao_pagamento,
        COUNT(DISTINCT fv.nk_pedido) AS qtd_pedidos,
        SUM(fv.valor_liquido)        AS receita_liquida
    FROM {CATALOG}.{SCHEMA_GOLD}.fato_vendas    fv
    JOIN {CATALOG}.{SCHEMA_GOLD}.dim_data       dd  ON fv.sk_data      = dd.sk_data
    JOIN {CATALOG}.{SCHEMA_GOLD}.dim_pagamento  dpg ON fv.sk_pagamento  = dpg.sk_pagamento
    GROUP BY dd.ano, dd.mes_nome, dd.mes, dpg.descricao_pagamento
    ORDER BY dd.ano, dd.mes, dpg.descricao_pagamento
"""))

In [0]:
# ===========================================================================
# CÉLULA 10 - Relatório final
# ===========================================================================
print("\n" + "="*50)
print("GOLD - Tabelas criadas:")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_GOLD}"))
